# Part 6 · Notebook 01 — Futures: fair value, rolls and continuous series

**Sessions:** S1 (Futures library) · [Lesson plan](../../docs/lessons/PART_06_FUTURES_OPTIONS_ENGINEERING.md) · graded labs in [`labs/part06/`](../../labs/part06/)

**You will:**
1. Price a future from spot and carry, and read the carry back from two contract months.
2. Stitch contracts into a continuous series three ways, and see what each preserves.
3. Keep orders on the real contract month, never on an adjusted price.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters, so every estimate can be compared with the truth. Units: T in years, σ as a decimal, vega per 1.00 σ, theta per year, `cp = +1` call / `−1` put.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p6lib.py is in notebooks/part06/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p6lib as p

p.use_course_style()

## 1. A futures strip

Quarterly index futures (March, June, September, December; expiry on the third Friday), each priced as spot × e^{carry·T} with a known carry of **3%** a year, a market in contango. Each contract trades for about six months before it expires.

In [ ]:
prices, spot, expiries, rolls = p.futures_panel()
ax = prices.plot(figsize=(11, 4), lw=1, legend=False)
spot.plot(ax=ax, color="black", lw=0.8, label="spot")
ax.set_title("Each contract converges to spot at its expiry"); plt.show()
print("roll dates (8 business days before each expiry):", [d.date().isoformat() for d in rolls[:4]], "…")

## 2. Fair value and implied carry

Cost of carry: `F = S·e^{(r−q)T}`. Turn it around and two contract months on the same day tell you the carry the market is pricing: `(r − q) = ln(F_far / F_near) / (T_far − T_near)`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def fair_value(S, r, q, T):
    return ...                                    # ✍️

def implied_carry(f_near, f_far, t_near, t_far):
    return ...                                    # ✍️

day = pd.Timestamp("2024-04-15")
near, far = prices.loc[day].dropna().index[:2]
t = [(e - day.date()).days / 365 for e in expiries if f"ES{p.MONTH_CODES[e.month - 1]}{str(e.year)[-1]}" in (near, far)]
mine = [fair_value(5000.0, 0.05, 0.015, 0.25), implied_carry(prices.at[day, near], prices.at[day, far], t[0], t[1])]
mine = p.check("fair value and carry", mine, [p.fair_value(5000.0, 0.05, 0.015, 0.25),
                                             p.implied_carry(prices.at[day, near], prices.at[day, far], t[0], t[1])])
print(f"F = {mine[0]:.2f};  carry implied by {near}/{far} on {day.date()}: {mine[1]:.2%} (true 3.00%)")

## 3. One series from many contracts

Indicators need one long series. Holding the front contract and switching at each roll date (the **unadjusted** series) creates a jump at every roll: the next contract trades higher in contango. **Back-adjusting** removes it by shifting all history *before* each roll:
* **difference:** add `new − old` (both on the roll date) to every earlier price;
* **ratio:** multiply every earlier price by `new / old`.

The latest contract's prices never change. `rolls[i]` is the day the strategy switches from column `i` to column `i + 1`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def back_adjust_difference(prices, rolls):
    out = p.continuous(prices, rolls, "none")     # the unadjusted series
    cols = list(prices.columns)
    for i, d in enumerate(rolls):
        gap = ...                                 # ✍️ new contract minus old contract, on the roll date
        out[out.index < d] += gap
    return out

mine = p.attempt(back_adjust_difference, prices, rolls)
mine = p.check("difference back-adjustment", mine, p.continuous(prices, rolls, "difference"))

In [ ]:
series = {m: p.continuous(prices, rolls, m) for m in ("none", "difference", "ratio")}
fig, ax = plt.subplots(figsize=(11, 4))
for m, s in series.items():
    ax.plot(s.index, s, lw=1, label=m)
for d in rolls:
    ax.axvline(d, color="#e6e5e0", lw=1)
ax.set_title("Unadjusted vs back-adjusted continuous series (grey lines: rolls)"); ax.legend(); plt.show()

roll_days = pd.DatetimeIndex(rolls)
rows = {}
for m, s in series.items():
    pts = s.diff()
    rows[m] = {"mean change on roll days": pts.loc[roll_days].mean(),
               "mean change on other days": pts.drop(roll_days).mean(),
               "total change over the sample": s.iloc[-1] - s.iloc[0],
               "first price": s.iloc[0]}
display(pd.DataFrame(rows).T.round(2))
gaps = [prices.at[d, prices.columns[i + 1]] - prices.at[d, prices.columns[i]] for i, d in enumerate(rolls)]
print(f"contango gaps at the {len(rolls)} rolls: {np.round(gaps, 2)} → {sum(gaps):.0f} points in total")

Unadjusted, every roll day adds the contango gap to the series: about +30 points on top of the day's real move. Nobody earned it, yet a return, momentum or breakout calculation counts it, and over the sample the unadjusted series overstates the P&L of actually holding the front contract by the sum of the gaps. Difference adjustment keeps **point changes** exactly (the P&L of one contract), ratio adjustment keeps **percentage returns**. Both change the level of old prices, so an adjusted price is never a price you can trade.

## 4. Orders go to the real contract

Signals come from the adjusted series; orders must name the contract month actually held on that day. With `rolls` sorted, the held column index is the number of roll dates on or before the day.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def held_contract(day, prices, rolls):
    i = ...                                       # ✍️ how many roll dates are on or before `day`
    return prices.columns[i]

days = [pd.Timestamp(d) for d in ("2024-01-10", "2024-03-05", "2024-03-06", "2024-12-31", "2025-11-28")]
mine = [p.attempt(held_contract, d, prices, rolls) for d in days]
expected = [prices.columns[int(np.searchsorted(pd.DatetimeIndex(rolls), d, side="right"))] for d in days]
mine = p.check("held_contract", mine, expected)
list(zip([d.date().isoformat() for d in days], mine))

## Wrap-up

* Fair value from carry, and carry from the curve.
* Signals on a back-adjusted series (difference for point P&L, ratio for returns); orders on the actual contract month.
* Graded version: `labs/part06/week21_futures_chains` (roll rules including volume crossover, all three continuous methods, tested for what each preserves).